In [0]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import xml.etree.ElementTree as ET
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://poc-sapdevene.enesis.com:8020/sap/opu/odata/sap/ZCDS_KNA1_ODATA_CDS/ZCDS_KNA1_ODATA"
username = dbutils.secrets.get(scope="sap-odata-username", key="client_secret")
password = dbutils.secrets.get(scope="sap-odata-password", key="client_secret")

headers = {
    "Accept": "application/atom+xml",  # eksplisit minta XML
    "sap-client": "120"
}

params = {
    "$filter": "Erdat ge datetime'2019-01-01T00:00:00' and Erdat le datetime'2019-12-31T23:59:59'",
}

response = requests.get(
    url,
    auth=HTTPBasicAuth(username, password),
    headers=headers,
    params=params,
    verify=False,
    timeout=60
)
print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Response length: {len(response.content)} bytes")
print("---RAW---")
print(response.text)

# Parse XML
try:    root = ET.fromstring(response.content)
except ET.ParseError as e:
    raise RuntimeError(f"Failed to parse XML: {e}")

# Parse XML
root = ET.fromstring(response.content)

ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "m": "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
    "d": "http://schemas.microsoft.com/ado/2007/08/dataservices"
}

records = []

# Cek apakah root adalah <feed> (collection) atau <entry> (single)
root_tag = root.tag.split("}")[-1]

if root_tag == "feed":
    # Collection — loop semua entry
    for entry in root.findall("atom:entry", ns):
        props = entry.find(".//m:properties", ns)
        if props is not None:
            row = {child.tag.split("}")[-1]: child.text for child in props}
            records.append(row)

elif root_tag == "entry":
    # Single entity — langsung ambil properties dari root
    props = root.find(".//m:properties", ns)
    if props is not None:
        row = {child.tag.split("}")[-1]: child.text for child in props}
        records.append(row)

df = pd.DataFrame(records)
print(f"Total rows: {len(df):,}")
print(df.T)  # .T biar keliatan semua kolom (karena banyak field)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StringType

# ============================================================
# KNA1 Schema — SAP OData → Databricks Bronze
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    NullType, StringType, DateType, 
    DecimalType, IntegerType, LongType, TimestampType
)
from pyspark.sql.functions import col, to_date, current_timestamp, lit

kna1_schema = {
    "Kunnr": StringType(),
    "Land1": StringType(),
    "Name1": StringType(),
    "Name2": StringType(),
    "Ort01": StringType(),
    "Pstlz": StringType(),
    "Regio": StringType(),
    "Sortl": StringType(),
    "Stras": StringType(),
    "Telf1": StringType(),
    "Telfx": StringType(),
    "Xcpdk": StringType(),
    "Adrnr": StringType(),
    "Mcod1": StringType(),
    "Mcod2": StringType(),
    "Mcod3": StringType(),
    "Anred": StringType(),
    "Aufsd": StringType(),
    "Bahne": StringType(),
    "Bahns": StringType(),
    "Bbbnr": StringType(),
    "Bbsnr": StringType(),
    "Begru": StringType(),
    "Brsch": StringType(),
    "Bubkz": StringType(),
    "Datlt": StringType(),
    "Erdat": DateType(),
    "Ernam": StringType(),
    "Exabl": StringType(),
    "Faksd": StringType(),
    "Fiskn": StringType(),
    "Knazk": StringType(),
    "Knrza": StringType(),
    "Konzs": StringType(),
    "Ktokd": StringType(),
    "Kukla": StringType(),
    "Lifnr": StringType(),
    "Lifsd": StringType(),
    "Locco": StringType(),
    "Loevm": StringType(),
    "Name3": StringType(),
    "Name4": StringType(),
    "Niels": StringType(),
    "Ort02": StringType(),
    "Pfach": StringType(),
    "Pstl2": StringType(),
    "Counc": StringType(),
    "Cityc": StringType(),
    "Rpmkr": StringType(),
    "Sperr": StringType(),
    "Spras": StringType(),
    "Stcd1": StringType(),
    "Stcd2": StringType(),
    "Stkza": StringType(),
    "Stkzu": StringType(),
    "Telbx": StringType(),
    "Telf2": StringType(),
    "Teltx": StringType(),
    "Telx1": StringType(),
    "Lzone": StringType(),
    "Xzemp": StringType(),
    "Vbund": StringType(),
    "Stceg": StringType(),
    "Dear1": StringType(),
    "Dear2": StringType(),
    "Dear3": StringType(),
    "Dear4": StringType(),
    "Dear5": StringType(),
    "Gform": StringType(),
    "Bran1": StringType(),
    "Bran2": StringType(),
    "Bran3": StringType(),
    "Bran4": StringType(),
    "Bran5": StringType(),
    "Ekont": StringType(),
    "Umsat": DecimalType(23, 2),
    "Umjah": StringType(),
    "Uwaer": StringType(),
    "Jmzah": StringType(),
    "Jmjah": StringType(),
    "Katr1": StringType(),
    "Katr2": StringType(),
    "Katr3": StringType(),
    "Katr4": StringType(),
    "Katr5": StringType(),
    "Katr6": StringType(),
    "Katr7": StringType(),
    "Katr8": StringType(),
    "Katr9": StringType(),
    "Katr10": StringType(),
    "Stkzn": StringType(),
    "Umsa1": DecimalType(23, 2),
    "Txjcd": StringType(),
    "Periv": StringType(),
    "Abrvw": StringType(),
    "Inspbydebi": StringType(),
    "Inspatdebi": StringType(),
    "Ktocd": StringType(),
    "Pfort": StringType(),
    "Werks": StringType(),
    "Dtams": StringType(),
    "Dtaws": StringType(),
    "Duefl": StringType(),
    "Hzuor": StringType(),
    "Sperz": StringType(),
    "Etikg": StringType(),
    "Civve": StringType(),
    "Milve": StringType(),
    "Kdkg1": StringType(),
    "Kdkg2": StringType(),
    "Kdkg3": StringType(),
    "Kdkg4": StringType(),
    "Kdkg5": StringType(),
    "Xknza": StringType(),
    "Fityp": StringType(),
    "Stcdt": StringType(),
    "Stcd3": StringType(),
    "Stcd4": StringType(),
    "Stcd5": StringType(),
    "Stcd6": StringType(),
    "Xicms": StringType(),
    "Xxipi": StringType(),
    "Xsubt": StringType(),
    "Cfopc": StringType(),
    "Txlw1": StringType(),
    "Txlw2": StringType(),
    "Ccc01": StringType(),
    "Ccc02": StringType(),
    "Ccc03": StringType(),
    "Ccc04": StringType(),
    "BondedAreaConfirm": StringType(),
    "DonateMark": StringType(),
    "ConsolidateInvoice": StringType(),
    "AllowanceType": StringType(),
    "EinvoiceMode": StringType(),
    "Cassd": StringType(),
    "Knurl": StringType(),
    "J1kfrepre": StringType(),
    "J1kftbus": StringType(),
    "J1kftind": StringType(),
    "Confs": StringType(),
    "Updat": DateType(),
    "Uptim": StringType(),
    "Nodel": StringType(),
    "Dear6": StringType(),
    "CvpXblck": StringType(),
    "Suframa": StringType(),
    "Rg": StringType(),
    "Exp": StringType(),
    "Uf": StringType(),
    "Rgdate": DateType(),
    "Ric": StringType(),
    "Rne": StringType(),
    "Rnedate": DateType(),
    "Cnae": StringType(),
    "Legalnat": StringType(),
    "Crtn": StringType(),
    "Icmstaxpay": StringType(),
    "Indtyp": StringType(),
    "Tdt": StringType(),
    "Comsize": StringType(),
    "Decregpc": StringType(),
    "Kna1EewCust": StringType(),
    "xvsoxrPalhgt": DecimalType(15, 3),
    "xvsoxrPalUl": StringType(),
    "xvsoxrPkMat": StringType(),
    "xvsoxrMatpal": StringType(),
    "xvsoxrINoLyr": StringType(),
    "xvsoxrOneMat": StringType(),
    "xvsoxrOneSort": StringType(),
    "xvsoxrUldSide": StringType(),
    "xvsoxrLoadPref": StringType(),
    "xvsoxrDpoint": StringType(),
    "Alc": StringType(),
    "PmtOffice": StringType(),
    "FeeSchedule": StringType(),
    "Duns": StringType(),
    "Duns4": StringType(),
    "SamUeId": StringType(),
    "SamEftInd": StringType(),
    "Psofg": StringType(),
    "Psois": StringType(),
    "Pson1": StringType(),
    "Pson2": StringType(),
    "Pson3": StringType(),
    "Psovn": StringType(),
    "Psotl": StringType(),
    "Psohs": StringType(),
    "Psost": StringType(),
    "Psoo1": StringType(),
    "Psoo2": StringType(),
    "Psoo3": StringType(),
    "Psoo4": StringType(),
    "Psoo5": StringType(),
    "J1iexcd": StringType(),
    "J1iexrn": StringType(),
    "J1iexrg": StringType(),
    "J1iexdi": StringType(),
    "J1iexco": StringType(),
    "J1icstno": StringType(),
    "J1ilstno": StringType(),
    "J1ipanno": StringType(),
    "J1iexcicu": StringType(),
    "Aedat": DateType(),
    "Usnam": StringType(),
    "J1isern": StringType(),
    "J1ipanref": StringType(),
    "GstTds": StringType(),
    "J3getyp": StringType(),
    "J3greftyp": StringType(),
    "Pspnr": StringType(),
    "Coaufnr": StringType(),
    "J3gagext": StringType(),
    "J3gagint": StringType(),
    "J3gagdumi": StringType(),
    "J3gagstdi": StringType(),
    "Lgort": StringType(),
    "Kokrs": StringType(),
    "Kostl": StringType(),
    "J3gabglg": StringType(),
    "J3gabgvg": StringType(),
    "J3gabrart": StringType(),
    "J3gstdmon": DecimalType(15, 3),
    "J3gstdtag": DecimalType(15, 3),
    "J3gtagmon": DecimalType(15, 3),
    "J3gzugtag": StringType(),
    "J3gmaschb": StringType(),
    "J3gmeinsa": StringType(),
    "J3gkeinsa": StringType(),
    "J3gblsper": StringType(),
    "J3gkleivo": StringType(),
    "J3gcalid": StringType(),
    "J3gvmonat": StringType(),
    "J3gabrken": StringType(),
    "J3glabrech": DateType(),
    "J3gaabrech": DateType(),
    "J3gzutvhlg": StringType(),
    "J3gnegmen": StringType(),
    "J3gfristlo": StringType(),
    "J3geminbe": StringType(),
    "J3gfmgue": StringType(),
    "J3gzuschue": StringType(),
    "J3gschprs": StringType(),
    "J3ginvsta": StringType(),
    "xsapcemxdber": StringType(),
    "xsapcemxkvmeq": StringType(),
}

spark = SparkSession.builder.getOrCreate()

# Step 1: pandas df → Spark df
spark_df = spark.createDataFrame(df)

# Step 2: Cast schema
for col_name, col_type in kna1_schema.items():
    if col_name in spark_df.columns:
        if isinstance(col_type, DateType):
            # SAP OData kirim format ISO: "2019-08-23T00:00:00"
            spark_df = spark_df.withColumn(col_name, to_date(col(col_name), "yyyy-MM-dd'T'HH:mm:ss"))
        else:
            spark_df = spark_df.withColumn(col_name, col(col_name).cast(col_type))

# Step 3: Fix sisa kolom NullType
for field in spark_df.schema.fields:
    if isinstance(field.dataType, NullType):
        spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

# Step 4: Audit columns + write
spark_df = spark_df \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source", lit("ZCDS_KNA1_ODATA"))

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("poc_enesis.bronze.ZCDS_KNA1_ODATA")

print("✅ Bronze table berhasil dibuat!")
spark.sql("SELECT COUNT(*) as total FROM poc_enesis.bronze.ZCDS_KNA1_ODATA").show()

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, to_date

watermark_row = spark.createDataFrame([Row(
    source_name   = "SAP_ODATA",
    table_name    = "ZCDS_KNA1_ODATA",
    last_run_date = RUN_DATE[:10],
    last_run_ts   = None,
    status        = "SUCCESS",
    rows_ingested = int(spark_df.count())
)])

watermark_row = watermark_row \
    .withColumn("last_run_date", to_date("last_run_date")) \
    .withColumn("last_run_ts", current_timestamp())

from delta.tables import DeltaTable
wm = DeltaTable.forName(spark, "poc_enesis.bronze.pipeline_watermark")
wm.alias('t').merge(
    watermark_row.alias('s'),
    't.source_name = s.source_name AND t.table_name = s.table_name'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ Watermark updated")


In [0]:
import uuid
from datetime import datetime

run_id    = str(uuid.uuid4())[:8]
started   = datetime.utcnow()
SCHEMA     = 'BRONZE'
TABLE     = 'ZCDS_KNA1_ODATA'

try:
    # ... semua logic ingest & write ke bronze ...
    rows_in = spark_df.count()
    status  = 'SUCCESS'
    msg     = f'{rows_in} rows ingested'
except Exception as e:
    rows_in = 0
    status  = 'FAILED'
    msg     = str(e)[:500]
    raise  # re-raise supaya Job tau pipeline gagal
finally:
    ended = datetime.utcnow()
    log_df = spark.createDataFrame([{
        'run_id': run_id, 'schema_name': SCHEMA, 'table_name': TABLE,
        'status': status, 'message': msg, 'rows_in': rows_in, 'rows_out': 0,
        'started_at': started, 'ended_at': ended
    }])
    log_df.write.format('delta').mode('append').saveAsTable(
        'poc_enesis.bronze.run_log'
    )
